In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile


In [ ]:
def unzip_dataset(zip_path, target_path):
    if not os.path.exists(zip_path):
        print(f" Файл {zip_path} не знайдено. Перевірте, чи ви завантажили його в Colab.")
        return

    if not os.path.exists(target_path):
        os.makedirs(target_path)
        print(f"--- Розпакування архіву у {target_path} ---")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_path)
        print(" Розпакування завершено успішно!")
    else:
        print(f"ℹ️ Папка {target_path} вже існує. Пропускаємо розпакування.")

In [ ]:
base_path = '/content/MMASH_Dataset/motion-and-heart-rate-from-a-wrist-worn-wearable-and-labeled-sleep-from-polysomnography-1.0.0'

In [ ]:
zip_file_path = '/content/motion-and-heart-rate-from-a-wrist-worn-wearable-and-labeled-sleep-from-polysomnography-1.0.0.zip'
destination_path = '/content/MMASH_Dataset'

In [ ]:
unzip_dataset(zip_file_path, destination_path)

In [ ]:
def prepare_dataset_final(root_path):
    # Визначаємо шляхи до папок згідно з вашою структурою
    hr_dir = os.path.join(root_path, 'heart_rate')
    labels_dir = os.path.join(root_path, 'labels')
    motion_dir = os.path.join(root_path, 'motion')
    steps_dir = os.path.join(root_path, 'steps')

    if not os.path.exists(labels_dir):
        print(f"Помилка: Папка {labels_dir} не знайдена!")
        return None

    label_files = [f for f in os.listdir(labels_dir) if f.endswith('_labeled_sleep.txt')]
    all_data = []
    G_MS2 = 9.80665

    for l_file in label_files:
        user_id = l_file.split('_')[0]
        print(f"-> Обробка користувача {user_id}...")

        # Формуємо шляхи до всіх файлів для цього ID
        f_steps = os.path.join(steps_dir, f"{user_id}_steps.txt")
        f_hr = os.path.join(hr_dir, f"{user_id}_heartrate.txt")
        f_acc = os.path.join(motion_dir, f"{user_id}_acceleration.txt")
        f_sleep = os.path.join(labels_dir, l_file)

        # ПЕРЕВІРКА: чи є файл кроків (обов'язково) та інші файли
        if not os.path.exists(f_steps):
            print(f"   ! Файл кроків ВІДСУТНІЙ. Пропускаємо.")
            continue

        if not all(os.path.exists(f) for f in [f_hr, f_acc, f_sleep]):
            print(f"   ! Відсутні інші компоненти (HR/ACC). Пропускаємо.")
            continue

        try:
            # 1. Сон (Мітки)
            sleep = pd.read_csv(f_sleep, sep=r'\s+', header=None, names=['time', 'sleep_stage'])
            sleep['time_bin'] = (sleep['time'] // 30) * 30

            # 2. Кроки (Обов'язково)
            steps = pd.read_csv(f_steps, sep=',', header=None, names=['time', 'steps'])
            steps['time_bin'] = (steps['time'] // 30) * 30
            steps_30s = steps.groupby('time_bin')['steps'].max().reset_index()

            # 3. Пульс
            hr = pd.read_csv(f_hr, sep=',', header=None, names=['time', 'heart_rate'])
            hr['time_bin'] = (hr['time'] // 30) * 30
            hr_30s = hr.groupby('time_bin')['heart_rate'].mean().reset_index()

            # 4. Акселерометр (Mobility)
            acc = pd.read_csv(f_acc, sep=r'\s+', header=None, names=['time', 'x', 'y', 'z'])
            acc['mobility'] = np.abs(np.sqrt(acc['x']**2 + acc['y']**2 + acc['z']**2) - 1.0) * G_MS2
            acc['time_bin'] = (acc['time'] // 30) * 30
            acc_30s = acc.groupby('time_bin')['mobility'].mean().reset_index()

            # Об'єднання (inner join гарантує, що в кожному рядку будуть і кроки, і пульс, і мітка)
            merged = sleep.merge(steps_30s, on='time_bin', how='inner') \
                          .merge(hr_30s, on='time_bin', how='inner') \
                          .merge(acc_30s, on='time_bin', how='inner')

            if not merged.empty:
                merged['user_id'] = user_id
                all_data.append(merged)
                print(f"   + Успішно зібрано {len(merged)} епох.")
            else:
                print(f"   ! Після злиття даних не залишилося (невідповідність часу).")

        except Exception as e:
            print(f"   - Помилка при читанні даних {user_id}: {e}")

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        # Фільтруємо невідомі стадії (наприклад, -1)
        final_df = final_df[final_df['sleep_stage'] >= 0]

        final_df.to_csv('processed_sleep_dataset.csv', index=False)
        print(f"\nГОТОВО! Разом {len(all_data)} користувачів. Збережено в 'processed_sleep_dataset.csv'")
        return final_df
    else:
        print("Не вдалося зібрати жодного повного набору даних.")
        return None


In [ ]:
# Запуск обробки
df = prepare_dataset_final(base_path)

In [ ]:
df = pd.read_csv("final_sleep_dataset.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'final_sleep_dataset.csv'

In [ ]:
df

,time,sleep_stage,time_bin,steps,heart_rate,mobility,user_id
0,180,0,180,0,69.000000,0.031579,4018081
1,780,1,780,0,59.000000,0.044681,4018081
2,1380,2,1380,0,58.833333,0.035279,4018081
3,1980,3,1980,0,59.333333,0.034100,4018081
4,2580,3,2580,0,59.166667,0.052949,4018081
...,...,...,...,...,...,...,...
842,25230,2,25230,0,68.166667,0.037982,844359
843,25830,5,25830,0,66.500000,0.039867,844359
844,26430,5,26430,0,63.666667,0.026416,844359
845,27030,5,27030,0,65.666667,0.052564,844359


In [ ]:
df.to_csv("all_in_mmash_raw.csv")

NameError: name 'df' is not defined

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 826 entries, 0 to 846
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   time         826 non-null    int64  
 1   sleep_stage  826 non-null    int64  
 2   time_bin     826 non-null    int64  
 3   steps        826 non-null    int64  
 4   heart_rate   826 non-null    float64
 5   mobility     826 non-null    float64
 6   user_id      826 non-null    object 
dtypes: float64(2), int64(4), object(1)
memory usage: 51.6+ KB


In [ ]:
import pandas as pd
import numpy as np
import os

def prepare_raw_for_comparison(root_path, output_path="mmash_for_interpolation_test.csv"):
    motion_dir = os.path.join(root_path, 'motion')
    hr_dir = os.path.join(root_path, 'heart_rate')
    steps_dir = os.path.join(root_path, 'steps')
    labels_dir = os.path.join(root_path, 'labels')

    label_files = [f for f in os.listdir(labels_dir) if f.endswith('_labeled_sleep.txt')]

    if os.path.exists(output_path):
        os.remove(output_path)

    first_user = True

    for l_file in label_files:
        user_id = l_file.split('_')[0]
        print(f"-> Обробка {user_id} (зберігаємо NaN для пульсу)...")

        try:
            # 1. Завантаження даних
            acc = pd.read_csv(os.path.join(motion_dir, f"{user_id}_acceleration.txt"),
                              sep=r'\s+', header=None, names=['time', 'x', 'y', 'z'])
            acc['time_dt'] = pd.to_timedelta(acc['time'], unit='s')

            hr = pd.read_csv(os.path.join(hr_dir, f"{user_id}_heartrate.txt"),
                             sep=',', header=None, names=['time', 'heart_rate'])
            hr['time_dt'] = pd.to_timedelta(hr['time'], unit='s')

            f_steps = os.path.join(steps_dir, f"{user_id}_steps.txt")
            steps = pd.read_csv(f_steps, sep=',', header=None, names=['time', 'steps']) if os.path.exists(f_steps) else pd.DataFrame(columns=['time_dt', 'steps'])
            if not steps.empty: steps['time_dt'] = pd.to_timedelta(steps['time'], unit='s')

            sleep = pd.read_csv(os.path.join(labels_dir, l_file),
                                sep=r'\s+', header=None, names=['time', 'sleep_stage'])
            sleep['time_dt'] = pd.to_timedelta(sleep['time'], unit='s')

            # 2. ОБ'ЄДНАННЯ (як у твоєму коді, але з контролем пропусків)
            # ВАЖЛИВО: tolerance=pd.Timedelta('1s') гарантує, що якщо пульсу немає в межах секунди,
            # ми отримаємо NaN, а не копію старого значення.
            merged = pd.merge_asof(acc.sort_values('time_dt'),
                                   hr[['time_dt', 'heart_rate']].sort_values('time_dt'),
                                   on='time_dt', direction='backward',
                                   tolerance=pd.Timedelta('1s'))

            merged = pd.merge_asof(merged, sleep[['time_dt', 'sleep_stage']].sort_values('time_dt'),
                                   on='time_dt', direction='backward')

            # Кроки заповнюємо 0, бо крок або є, або його нема (інтерполяція тут зазвичай не потрібна)
            if not steps.empty:
                merged = pd.merge_asof(merged, steps[['time_dt', 'steps']].sort_values('time_dt'),
                                       on='time_dt', direction='backward', tolerance=pd.Timedelta('1s'))
                merged['steps'] = merged['steps'].fillna(0)
            else:
                merged['steps'] = 0

            # ТУТ МИ НЕ РОБИМО INTERPOLATE ДЛЯ HEART_RATE!

            merged['user_id'] = user_id
            merged.drop(columns=['time_dt'], inplace=True)

            # 3. Запис у файл
            if first_user:
                merged.to_csv(output_path, index=False, mode='w')
                first_user = False
            else:
                merged.to_csv(output_path, index=False, mode='a', header=False)

            del acc, hr, steps, sleep, merged # Звільняємо RAM

        except Exception as e:
            print(f"   ! Помилка у {user_id}: {e}")

    print(f"\n✅ ГОТОВО! Створено 'дирявий' датасет: {output_path}")

# Запуск (переконайся що base_path визначено)
prepare_raw_for_comparison(base_path)

-> Обробка 4018081 (зберігаємо NaN для пульсу)...
-> Обробка 8173033 (зберігаємо NaN для пульсу)...
-> Обробка 5498603 (зберігаємо NaN для пульсу)...
-> Обробка 759667 (зберігаємо NaN для пульсу)...
-> Обробка 1455390 (зберігаємо NaN для пульсу)...
-> Обробка 5797046 (зберігаємо NaN для пульсу)...
-> Обробка 1818471 (зберігаємо NaN для пульсу)...
-> Обробка 1449548 (зберігаємо NaN для пульсу)...
-> Обробка 1360686 (зберігаємо NaN для пульсу)...
-> Обробка 8530312 (зберігаємо NaN для пульсу)...
-> Обробка 5383425 (зберігаємо NaN для пульсу)...
-> Обробка 4426783 (зберігаємо NaN для пульсу)...
-> Обробка 3997827 (зберігаємо NaN для пульсу)...
-> Обробка 8686948 (зберігаємо NaN для пульсу)...
-> Обробка 4314139 (зберігаємо NaN для пульсу)...
-> Обробка 9961348 (зберігаємо NaN для пульсу)...
-> Обробка 1066528 (зберігаємо NaN для пульсу)...
-> Обробка 781756 (зберігаємо NaN для пульсу)...
-> Обробка 8692923 (зберігаємо NaN для пульсу)...
-> Обробка 6220552 (зберігаємо NaN для пульсу)...
->